离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch07_statistics_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第7章：估计、推断与bootstrap

仅用构造样本，每次观察的单位为单个构造期间收益。主种子20261007，五条子随机流分开用于样本、覆盖率、AR、IID和块重采样。

先比较Gaussian MLE方差与ddof=1方差、Student-t区间和p值，再模拟覆盖率，最后在AR(1)的历史训练片上比较IID与循环移动块bootstrap。固定块长只作教学选择。

## 继承的固定种子结果

40观察样本均值−.001500，MLE方差.00045812、样本方差.00046987；95% t区间[−.008432,.005432]，双侧p=.664051。3000重复中n=10/30/100的覆盖率为95.33%/94.93%/95.50%。AR训练片n=180、phi=.70，IID百分位区间[−.002327,.001643]，固定12长度块区间[−.003428,.002961]。这些是继承的构造运行记录，本次语言修订未重算。

## 1. 共用实现

估计器、检验、模拟和重采样来自 `src/quant_course/statistics_tools.py`，Notebook不另写一套实现。

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from quant_course.statistics_tools import (
    beta_binomial_posterior,
    bootstrap_mean_confidence_interval,
    bootstrap_mean_draws,
    mean_t_confidence_interval,
    moving_block_bootstrap_mean_draws,
    normal_mle_estimates,
    one_sample_t_test,
    simulate_stationary_ar1,
    simulate_t_interval_coverage,
)

SEED = 20261007
PERIOD_UNIT = "return per constructed observation period"
seed_sequence = np.random.SeedSequence(SEED)
normal_seed, coverage_seed, ar_seed, iid_seed, block_seed = seed_sequence.spawn(5)
normal_rng = np.random.default_rng(normal_seed)
coverage_rng = np.random.default_rng(coverage_seed)
ar_rng = np.random.default_rng(ar_seed)
iid_rng = np.random.default_rng(iid_seed)
block_rng = np.random.default_rng(block_seed)


## 2. MLE、无偏样本方差与t区间

Gaussian方差MLE把离均差平方和除以 $n$；通常无偏估计除以 $n-1$。均值t区间采用样本标准差与 $n-1$ 自由度。

In [ ]:
sample_n = 40
assumed_teaching_mean = 0.002
assumed_teaching_std = 0.02
normal_sample = normal_rng.normal(assumed_teaching_mean, assumed_teaching_std, size=sample_n)
estimates = normal_mle_estimates(normal_sample)
interval = mean_t_confidence_interval(normal_sample, confidence_level=0.95)
test = one_sample_t_test(normal_sample, null_mean=0.0)

print(f"CONSTRUCTED normal sample: n={sample_n}, seed={SEED}, unit={PERIOD_UNIT}")
print(f"sample mean={estimates.mean:.6f}")
print(f"MLE variance (ddof=0)={estimates.variance_mle:.8f}; unbiased variance (ddof=1)={estimates.variance_unbiased:.8f}")
print(f"95% Student-t mean CI=[{interval.lower:.6f}, {interval.upper:.6f}] {PERIOD_UNIT}; df={sample_n - 1}")
print(f"two-sided t={test.t_statistic:.6f}, p={test.p_value:.6f}")
print("The p-value is a tail probability under H0 and the test assumptions; it is not the probability that H0 is true.")

hand_check = normal_mle_estimates([1.0, 2.0, 3.0])
assert np.isclose(hand_check.variance_mle, 2.0 / 3.0)
assert hand_check.variance_unbiased == 1.0
assert interval.lower < interval.estimate < interval.upper


## 3. Beta–Binomial更新

先验Beta(2,2)，6成功、4失败后为Beta(8,6)，后验均值4/7。它只估计成功概率；收益大小、损失大小和期望价值仍未知。

In [ ]:
posterior = beta_binomial_posterior(6, 4, prior_alpha=2.0, prior_beta=2.0)
print(f"prior Beta({posterior.prior_alpha:g}, {posterior.prior_beta:g}); successes={posterior.successes}; failures={posterior.failures}")
print(f"posterior Beta({posterior.posterior_alpha:g}, {posterior.posterior_beta:g}); posterior mean={posterior.posterior_mean:.6f}")
assert (posterior.posterior_alpha, posterior.posterior_beta) == (8.0, 6.0)
assert np.isclose(posterior.posterior_mean, 4.0 / 7.0)


## 4. 重复正态抽样的区间覆盖率

每次独立抽样后，记录95% Student-t区间是否覆盖已知总体均值。各n重复3000次，有限MonteCarlo误差使观察比例偏离95%。

In [ ]:
coverage_repetitions = 3000
coverage_sizes = [10, 30, 100]
true_mean = 0.001
true_std = 0.02
coverage = simulate_t_interval_coverage(
    coverage_sizes,
    repetitions=coverage_repetitions,
    true_mean=true_mean,
    true_std=true_std,
    confidence_level=0.95,
    rng=coverage_rng,
)
print(f"CONSTRUCTED normal coverage simulation: true mean={true_mean}, true sd={true_std} {PERIOD_UNIT}")
print(f"seed={SEED}; repetitions per n={coverage_repetitions}; nominal coverage=95%; observed rates={coverage}")


## 5. 依赖序列上的IID与移动块bootstrap

构造240观察、phi=.70的平稳Gaussian AR(1)，仅前180行进入两种重采样，后60行保持留出。各重采样4000次。循环块保留局部顺序并截为训练长度；固定块长12不保证一般有效，仍需平稳/弱依赖条件及合理块长选择。n(1−phi)/(1+phi)只是该模型下有效样本量近似。

In [ ]:
full_n = 240
train_n = 180
phi = 0.70
innovation_std = 0.01
block_length = 12
bootstrap_repetitions = 4000
ar_series = simulate_stationary_ar1(full_n, phi=phi, innovation_std=innovation_std, rng=ar_rng)
training = ar_series[:train_n]
held_out = ar_series[train_n:]
iid_draws = bootstrap_mean_draws(training, n_resamples=bootstrap_repetitions, rng=iid_rng)
block_draws = moving_block_bootstrap_mean_draws(
    training,
    block_length=block_length,
    n_resamples=bootstrap_repetitions,
    rng=block_rng,
)
iid_ci = bootstrap_mean_confidence_interval(iid_draws)
block_ci = bootstrap_mean_confidence_interval(block_draws)
effective_n_approx = train_n * (1 - phi) / (1 + phi)
print(f"CONSTRUCTED stationary AR(1): n={full_n}, phi={phi:.2f}, innovation sd={innovation_std:.3f} {PERIOD_UNIT}")
print(f"training n={len(training)}, held-out n={len(held_out)}; bootstrap repetitions={bootstrap_repetitions}; fixed block length={block_length}")
print(f"approximate effective sample size n(1-phi)/(1+phi)={effective_n_approx:.2f} (heuristic under AR(1) assumptions)")
print(f"IID percentile 95% CI=[{iid_ci.lower:.6f}, {iid_ci.upper:.6f}] {PERIOD_UNIT}")
print(f"moving-block percentile 95% CI=[{block_ci.lower:.6f}, {block_ci.upper:.6f}] {PERIOD_UNIT}")
assert iid_ci.lower < iid_ci.upper
assert block_ci.lower < block_ci.upper


## 6. 图形核对

左图按样本量画覆盖比例及95%参考线，右图比较训练均值的两种bootstrap分布。末格新保存组合图；差异说明重采样假设影响不确定性，不证明某一种对所有金融序列有效。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sizes = np.array(list(coverage))
rates = np.array([coverage[size] for size in sizes])
axes[0].plot(sizes, rates * 100.0, marker="o", color="#235789")
axes[0].axhline(95.0, color="#C1292E", linestyle="--", label="Nominal 95%")
axes[0].set_xlabel("Sample size n (constructed observations)")
axes[0].set_ylabel("Intervals covering true mean (%)")
axes[0].set_title(f"Student-t interval coverage; {coverage_repetitions:,} simulations")
axes[0].set_ylim(85, 100)
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].hist(iid_draws, bins=45, density=True, alpha=0.52, label="IID bootstrap")
axes[1].hist(block_draws, bins=45, density=True, alpha=0.52, label=f"Moving blocks (L={block_length})")
axes[1].axvline(float(iid_draws.mean()), color="#235789", linestyle=":", linewidth=1.5)
axes[1].set_xlabel("Bootstrap mean (return per constructed period)")
axes[1].set_ylabel("Density (inverse return unit)")
axes[1].set_title(f"AR(1) training n={train_n}, phi={phi:.2f}; {bootstrap_repetitions:,} draws")
axes[1].grid(alpha=0.2)
axes[1].legend(frameon=False)
fig.suptitle(f"Chapter 7 — constructed simulation only; master seed {SEED}; no empirical performance inference")
fig.tight_layout()
figure_path = RUN_DIR / "figures" / "a2" / "ch07_statistics.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(figure_path)))
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## 检查与适用边界

[1,2,3]的MLE方差2/3，无偏方差1；Beta更新均值4/7。接口拒绝NaN、无穷、复数、非一维及过短样本。常数样本可算描述方差，但因标准误为零不能用于t推断。p值以零假设及检验条件为前提，不是零假设为真的概率。第一/二类错误与功效取决于预先声明的重复抽样设计，多重比较须定义检验族。IID抽样破坏序列顺序；块抽样需弱依赖、平稳和敏感性检查。成功概率模型本身不含盈亏金额。